# Annotate Restrictions

This notebook flags recipes with restriction labels based on parsed ingredient names.

In [ ]:
import ast
import json
import time
import pandas as pd
from pathlib import Path
from ingredient_parser import parse_ingredient

CSV_PATH = Path("../data/dataset_10000_normalized.csv")
OUT_PATH = Path("../data/dataset_10000_annotated.csv")
CHUNK_SIZE = 2000

In [2]:
# Restriction rules: ingredient keyword -> triggers this restriction
# Logic: if any parsed ingredient name contains any keyword -> recipe is flagged for that restriction

RULES_PATH = Path("../data/restriction_rules.json")
with RULES_PATH.open("r", encoding="utf-8") as f:
    RULES = json.load(f)

print(f"Loaded rules for {len(RULES)} restrictions")

Loaded rules for 72 restrictions


In [3]:
def parse_ingredient_names(raw_list: list[str]) -> list[str]:
    """Extract item names from raw ingredient strings using ingredient_parser."""
    names = []
    for raw in raw_list:
        try:
            parsed = parse_ingredient(raw)
            item = parsed.name[0].text if parsed.name else None
            if item:
                names.append(item.strip().lower())
        except Exception:
            pass
    return names


def annotate_row(ingredient_names: list[str]) -> list[str]:
    """Return list of triggered restrictions for a recipe."""
    text = " ".join(ingredient_names)  # single string for fast substring search
    triggered = []
    for restriction, keywords in RULES.items():
        if any(kw in text for kw in keywords):
            triggered.append(restriction)
    return triggered

In [4]:
# Smoke test on a few recipes
samples = [
    ["2 lb. crab meat", "1 cup mayo", "1 lemon"],
    ["1 lb. ground beef", "1 cup flour", "2 eggs", "1 cup milk"],
    ["1 cup almond flour", "1/2 cup honey", "2 eggs", "1 tsp vanilla"],
]

for raw_list in samples:
    names = parse_ingredient_names(raw_list)
    restrictions = annotate_row(names)
    print(f"Ingredients: {names}")
    print(f"Restrictions: {restrictions}")
    print()

Ingredients: ['crab meat', 'mayo', 'lemon']
Restrictions: ['egg_allergy', 'shellfish_allergy', 'vegan', 'vegetarian', 'lacto_vegetarian', 'ovo_vegetarian', 'lacto_ovo_vegetarian', 'kosher', 'hindu_vegetarian', 'jain', 'buddhist_vegetarian', 'pku_diet', 'no_shellfish']

Ingredients: ['ground beef', 'flour', 'eggs', 'milk']
Restrictions: ['milk_allergy', 'egg_allergy', 'wheat_allergy', 'alpha_gal_syndrome', 'lactose_intolerance', 'gluten_intolerance', 'celiac_disease', 'fodmap_intolerance', 'vegan', 'vegetarian', 'lacto_vegetarian', 'ovo_vegetarian', 'lacto_ovo_vegetarian', 'pescatarian', 'flexitarian', 'hindu_vegetarian', 'jain', 'buddhist_vegetarian', 'gluten_free', 'dairy_free', 'egg_free', 'low_fodmap', 'renal_diet', 'pku_diet', 'low_purine', 'aip_autoimmune_protocol', 'keto', 'paleo', 'low_carb', 'no_beef', 'no_red_meat']

Ingredients: ['almond flour', 'honey', 'eggs', 'vanilla']
Restrictions: ['nut_allergy', 'egg_allergy', 'wheat_allergy', 'gluten_intolerance', 'celiac_disease', 'f

## Annotation Workflow

Load rules, parse ingredient names, then assign restriction labels per recipe.

In [23]:
# Annotate the full dataset
results = []  # list of dicts: {index, restrictions}
total_rows = 0
t0 = time.time()

print("Annotating dataset...")
for chunk in pd.read_csv(
    CSV_PATH, usecols=["ingredients"], chunksize=CHUNK_SIZE, low_memory=False
):
    for val in chunk["ingredients"]:
        try:
            raw_list = ast.literal_eval(str(val))
        except Exception:
            raw_list = []
        names = parse_ingredient_names(raw_list)
        results.append(annotate_row(names))

    total_rows += len(chunk)
    elapsed = time.time() - t0
    print(f"  {total_rows:,} rows annotated  ({elapsed:.0f}s)")

print(f"\nDone in {time.time()-t0:.0f}s")

avg_restrictions = (sum(len(r) for r in results) / len(results)) if results else 0.0
print(f"Average restrictions per row: {avg_restrictions:.2f}")

Annotating dataset...
  10,000 rows annotated  (45s)

Done in 45s
Average restrictions per row: 22.30


In [24]:
# Merge annotations back into the full dataframe and save
df = pd.read_csv(CSV_PATH, low_memory=False)
df["exclusion_restrictions"] = [json.dumps(r) for r in results]

# Quick stats
from collections import Counter

all_flags = [r for row in results for r in row]
top = Counter(all_flags).most_common(15)
print("\nTop 15 most common restrictions:")
for restriction, count in top:
    print(f"  {restriction}: {count:,} recipes ({count/len(results)*100:.1f}%)")


Top 15 most common restrictions:
  aip_autoimmune_protocol: 9,020 recipes (90.2%)
  paleo: 8,638 recipes (86.4%)
  vegan: 8,411 recipes (84.1%)
  pku_diet: 7,848 recipes (78.5%)
  keto: 7,749 recipes (77.5%)
  low_carb: 7,492 recipes (74.9%)
  ovo_vegetarian: 7,366 recipes (73.7%)
  alpha_gal_syndrome: 6,906 recipes (69.1%)
  jain: 6,889 recipes (68.9%)
  low_fodmap: 6,822 recipes (68.2%)
  buddhist_vegetarian: 6,755 recipes (67.5%)
  fodmap_intolerance: 6,735 recipes (67.3%)
  renal_diet: 6,703 recipes (67.0%)
  milk_allergy: 6,431 recipes (64.3%)
  lactose_intolerance: 6,403 recipes (64.0%)


## Export

Save the annotated dataset and review summary stats.

In [25]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved annotated dataset to {OUT_PATH}")

Saved annotated dataset to ../data/dataset_10000_annotated.csv
